# Chapter 3

In [ ]:
# Section 3.3
import oandapyV20
from oandapyV20.endpoints.accounts import AccountDetails
from oandapyV20.endpoints.pricing import PricingStream

# Replace with your OANDA API key and account ID
api_key = 'YOUR_API_KEY'
account_id = 'YOUR_ACCOUNT_ID'

client = oandapyV20.API(access_token=api_key)

# Retrieve account details
account_details = AccountDetails(accountID)
response = client.request(account_details)
print(response)

# Stream real-time pricing data
params = {
    "instruments": "EUR_USD"
}
pricing_stream = PricingStream(accountID=accountID, params=params)
response = client.request(pricing_stream)
for tick in response:
    print(tick)

# Chapter 4

In [ ]:
# Section 1.1
import pandas as pd
df = pd.read_csv("BTCUSD_Candlestick_1_D_ASK_08.05.2017-16.10.2021.csv")
df=df[df['Volume']!=0]
df=df[df["High"]!=df["Low"]]
df=df.reset_index(drop=True)

In [ ]:
# Section 1.2.1
df['Gmt time'] = pd.to_datetime(df['Gmt time'], 
                                format='%d.%m.%Y %H:%M:%S.%f')
missing_values = df['Gmt time'].isnull().sum()
if missing_values == 0:
    print("No missing values in the GMT time column.")
else:
    print(f"There are {missing_values} missing values in the GMT time column.")

In [ ]:
# Section 1.2.2
is_increasing = df['Gmt time'].is_monotonic_increasing
if is_increasing:
    print("The GMT time values are clean and always increasing.")
else:
    print("The GMT time values are not always increasing.")

In [ ]:
non_increasing = df[df['Gmt time'] <= df['Gmt time'].shift(1)]
if len(non_increasing) > 0:
    print("The following rows have non-increasing GMT time values:")
    print(non_increasing)
else:
    print("All GMT time values are correctly increasing.")

In [ ]:
df['time_diff'] = df['Gmt time'].diff().dt.total_seconds()
df['time_diff'].plot(kind='line', title='Time Differences Between Consecutive GMT Values')

In [ ]:
import matplotlib.pyplot as plt

# Calculate the time differences in seconds
df['time_diff'] = df['Gmt time'].diff().dt.total_seconds()

# Convert the time differences from seconds to hours
df['time_diff_hours'] = df['time_diff'] / 3600

# Plot the time differences in hours
df['time_diff_hours'].plot(kind='line', title='Time Differences Between Consecutive GMT Values (in Hours)')
plt.ylabel('Hours')
plt.xlabel('Index')
plt.show()

In [ ]:
# Section 1.3.1
import yfinance as yf
data = yf.download(tickers='BTC-USD', period='1mo', interval='15m')
data = data[data["High"]!=data["Low"]]

In [ ]:
# Section 1.3.2
import yfinance as yf
# Download data for multiple tickers: Bitcoin, Ethereum, and Tesla
data = yf.download(tickers=['BTC-USD', 'ETH-USD', 'TSLA'], period='1mo', interval='1d')

In [ ]:
# Section 1.3.3
import yfinance as yf
# Download data for Bitcoin from January 1, 2021, to December 31, 2021
data = yf.download(tickers='BTC-USD', start='2021-01-01', end='2021-12-31', interval='1d')
data.head()

In [ ]:
# Section 1.3.4
import yfinance as yf
# Download 1-minute interval data for Bitcoin for the last 7 days
data = yf.download(tickers='BTC-USD', period='7d', interval='1m')
print(data.head())

In [ ]:
# Section 1.3.5
import yfinance as yf
# Download adjusted close prices and fill missing values forward
data = yf.download(tickers='BTC-USD', period='1mo', interval='1d', auto_adjust=True, prepost=True)
print(data.head())

In [ ]:
# Section 1.4
from oanda_candles import Pair, Gran, CandleClient
import pandas as pd

access_token = "XXX"  # Your OANDA API token
accountID = "XXX"     # Your OANDA account ID

def get_candles(n):
    client = CandleClient(access_token, real=False)
    collector = client.get_collector(Pair.EUR_USD, Gran.M5)
    return collector.grab(n)

def get_candles_frame(n):
    candles = get_candles(n)
    
    dfstream = pd.DataFrame([{
        'Open': float(candle.bid.o.value),
        'Close': float(candle.bid.c.value),
        'High': float(candle.bid.h.value),
        'Low': float(candle.bid.l.value)
    } for candle in candles])

    return dfstream

get_candles_frame(5)

In [ ]:
# Retrieve the candles data and save it to a CSV file
dfstream = get_candles_frame(5)
dfstream.to_csv('candles_data.csv', index=False)

In [ ]:
# Section 2.1
dfpl = data[0:100]
import matplotlib.pyplot as plt
plt.plot(dfpl.index, dfpl.Close) #,go
#color='green', marker='o', linestyle='dashed',linewidth=2, markersize=12
plt.show()

In [ ]:
# Section 2.2
dfpl = data[0:100]
import plotly.graph_objects as go

fig = go.Figure(data=[go.Candlestick(x=dfpl.index,
                open=dfpl['Open'],
                high=dfpl['High'],
                low=dfpl['Low'],
                close=dfpl['Close'])])

fig.show()

In [ ]:
dfpl = data[0:100]
import plotly.graph_objects as go
fig = go.Figure(data=go.Candlestick(x=dfpl.index,
                open=dfpl['Open'],
                high=dfpl['High'],
                low=dfpl['Low'],
                close=dfpl['Close'],
                increasing_line_color= 'yellow', decreasing_line_color= 'blue'))
fig.update_layout(paper_bgcolor="black", plot_bgcolor="black", 
                  margin_l=10, margin_b=0, margin_r=0, margin_t=0,
                 grid_columns=1,grid_rows=1)
fig.update_xaxes(showline=True, linewidth=2, linecolor='black', gridcolor='black')
fig.update_yaxes(showline=True, linewidth=2, linecolor='black', gridcolor='black')
fig.show()

In [ ]:
# Section 3.1
df.ta.indicators()
help(ta.rsi)

In [ ]:
# Section 3.2.1
import pandas_ta as ta
df["EMA"] = ta.ema(df.Close, length=10)

In [ ]:
dfpl = df[50:150]
import plotly.graph_objects as go

fig = go.Figure(data=[go.Candlestick(x=dfpl.index,
                open=dfpl['Open'],
                high=dfpl['High'],
                low=dfpl['Low'],
                close=dfpl['Close']),
                go.Scatter(x=dfpl.index, y=dfpl.EMA, line=dict(color='red', width=2), name="EMA")])

fig.show()

In [ ]:
# Section 3.2.2
import pandas_ta as ta
my_bbands = ta.bbands(data.Close, length=15, std=1.5)
data=data.join(my_bbands)

In [ ]:
import plotly.graph_objects as go

df = data[100:500]
fig = go.Figure()

# Candlestick chart
fig.add_trace(go.Candlestick(x=df.index,
                open=df['Open'],
                high=df['High'],
                low=df['Low'],
                close=df['Close']))

# Upper Bollinger Band (BBU)
fig.add_trace(go.Scatter(x=df.index, y=df['BBU_15_1.5'], name='BB_UPPER',
                         line=dict(color='blue')))

# Middle Bollinger Band (BBM)
fig.add_trace(go.Scatter(x=df.index, y=df['BBM_15_1.5'], name='BB_MIDDLE',
                         line=dict(color='blue')))

# Lower Bollinger Band (BBL)
fig.add_trace(go.Scatter(x=df.index, y=df['BBL_15_1.5'], name='BB_LOWER',
                         line=dict(color='blue')))

# Layout settings
fig.update_layout(
    xaxis=dict(rangeslider=dict(visible=False))
)

fig.show()

In [ ]:
# Section 3.3
import plotly.graph_objects as go
from plotly.subplots import make_subplots
fig = make_subplots(rows=2, cols=1, subplot_titles=['Price', 'ATR'], shared_xaxes=True)

fig.add_trace(go.Candlestick(x=df.index,
                open=df['Open'],
                high=df['High'],
                low=df['Low'],
                close=df['Close']), row=1, col=1)
fig.add_trace(go.Scatter(x=df.index, y=df['ATR'], name='ATR'), row=2, col=1)

fig.update_layout(
    xaxis=dict(rangeslider=dict(visible=False))
)

fig.show()

# Chapter 5

In [ ]:
# Section 2.1
def ema_signal(df, backcandles):
    # Create boolean Series for conditions
    above = df['EMA_fast'] > df['EMA_slow']
    below = df['EMA_fast'] < df['EMA_slow']

    # Rolling window to check if condition is met consistently over the window
    above_all = above.rolling(window=backcandles).apply(lambda x: x.all(), raw=True).fillna(0).astype(bool)
    below_all = below.rolling(window=backcandles).apply(lambda x: x.all(), raw=True).fillna(0).astype(bool)

    # Assign signals based on conditions
    df['EMASignal'] = 0  # Default no signal
    df.loc[above_all, 'EMASignal'] = 2  # Signal 2 where EMA_fast consistently above EMA_slow
    df.loc[below_all, 'EMASignal'] = 1  # Signal 1 where EMA_fast consistently below EMA_slow

    return df

import pandas_ta as ta
df["EMA_slow"]=ta.ema(df.Close, length=50)
df["EMA_fast"]=ta.ema(df.Close, length=30)


In [ ]:
# Section 2.2
def rsi_signal(df, backcandles=5):
    rsi_series = df['RSI']
    
    # Create boolean Series for conditions
    above_50 = rsi_series.gt(50.1)
    below_50 = rsi_series.lt(49.9)

    # Rolling window to check if condition is met consistently over the window
    above_all = above_50.rolling(window=backcandles).apply(lambda x: x.all(), raw=True).fillna(0).astype(bool)
    below_all = below_50.rolling(window=backcandles).apply(lambda x: x.all(), raw=True).fillna(0).astype(bool)

    # Initialize the RSISignal column in the DataFrame with 0 (no signal)
    df['RSISignal'] = 0

    # Assign signals based on conditions
    df.loc[above_all, 'RSISignal'] = 2  # Signal 2 where RSI consistently above 50.1
    df.loc[below_all, 'RSISignal'] = 1  # Signal 1 where RSI consistently below 49.9

    return df


In [ ]:
df = ema_signal(df, 7)
df = rsi_signal(df, 7)
df['TotalSignal'] = df.apply(lambda row: row['EMA_Signal'] 
                             if row['EMASignal'] == row['RSISignal'] 
                             else 0, axis=1)


In [ ]:
# Section 3.1
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression

# Example data: closing prices over time
data = {
    'time': np.arange(10),  # Time points (e.g., days)
    'close_price': [100, 102, 104, 103, 105, 
                    107, 106, 108, 110, 109]  # Example closing prices
}
df = pd.DataFrame(data)

# Reshape time data to fit the model
X = df['time'].values.reshape(-1, 1)  # Independent variable (time)
y = df['close_price'].values  # Dependent variable (price)

# Apply linear regression
model = LinearRegression()
model.fit(X, y)

# Get the slope of the fitted line
slope = model.coef_[0]

print(f'The slope of the trend is: {slope}')
# The slope of the trend is: 1.0181818181818183


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression

# Example data: closing prices over time
data = {
    'time': np.arange(10),  # Time points (e.g., days)
    'close_price': [100, 102, 104, 103, 105, 107, 106, 108, 110, 109]  # Example closing prices
}
df = pd.DataFrame(data)

# Reshape time data to fit the model
X = df['time'].values.reshape(-1, 1)  # Independent variable (time)
y = df['close_price'].values  # Dependent variable (price)

# Apply linear regression
model = LinearRegression()
model.fit(X, y)

# Get the slope and intercept of the fitted line
slope = model.coef_[0]
intercept = model.intercept_

# Predict the trend line
trend_line = model.predict(X)

# Plotting the data points and the trend line
plt.figure(figsize=(10, 6))
plt.scatter(df['time'], df['close_price'], color='blue', label='Data Points')
plt.plot(df['time'], trend_line, color='red', label=f'Trend Line (Slope: {slope:.2f})')
plt.title('Closing Prices with Trend Line')
plt.xlabel('Time')
plt.ylabel('Closing Price')
plt.legend()
plt.grid(True)
plt.show()


In [ ]:
# Section 3.2
import pandas as pd
import pandas_ta as ta
from sklearn.linear_model import LinearRegression
import numpy as np

# Function to calculate the slope of a series over a specified number of backcandles
def calculate_slope(series, backcandles=7, startindex=87):
    slopes = [np.nan] * startindex  # Initialize with NaN for the first rows
    
    for i in range(startindex, len(series)):
        # Select the range of data to compute the slope
        y = series[i-backcandles:i].values
        x = np.arange(backcandles).reshape(-1, 1)
        
        # Perform linear regression to get the slope
        model = LinearRegression()
        model.fit(x, y)
        slopes.append(model.coef_[0])
    
    return slopes

# Function to add EMA slopes to the DataFrame
def add_ema_slopes(df, backcandles):
    df['EMA_20'] = ta.ema(df['Close'], length=20)
    df['EMA_50'] = ta.ema(df['Close'], length=50)
    df['EMA_80'] = ta.ema(df['Close'], length=80)
    
    # Calculate slopes and add them as new columns
    df['Slope_EMA_20'] = calculate_slope(df['EMA_20'], backcandles)
    df['Slope_EMA_50'] = calculate_slope(df['EMA_50'], backcandles)
    df['Slope_EMA_80'] = calculate_slope(df['EMA_80'], backcandles)
    
    return df

# Example usage
df = pd.read_csv("BTCUSD_Candlestick_1_D_ASK_08.05.2017-16.10.2021.csv")
df = df[df['Volume'] != 0]
df = df[df["High"] != df["Low"]]
df = df.reset_index(drop=True)

# Add EMA slopes with a specified number of backcandles (e.g., 7)
df = add_ema_slopes(df, backcandles=7)

df.tail(15)


In [ ]:
# Section 4
def isPivot(candle, window):
    """
    function that detects if a candle is a pivot/fractal point
    args: candle index, window before and after candle to test if pivot
    returns: 1 if pivot high, 2 if pivot low, 3 if both and 0 default
    """
    if candle-window < 0 or candle+window >= len(df):
        return 0
    
    pivotHigh = 1
    pivotLow = 2
    for i in range(candle-window, candle+window+1):
        if df.iloc[candle].Low > df.iloc[i].Low:
            pivotLow=0
        if df.iloc[candle].High < df.iloc[i].High:
            pivotHigh=0
    if (pivotHigh and pivotLow):
        return 3
    elif pivotHigh:
        return pivotHigh
    elif pivotLow:
        return pivotLow
    else:
        return 0


In [ ]:
window=3
df['isPivot'] = df.apply(lambda x: isPivot(x.name,window), axis=1)


In [ ]:
def add_pointpos_column(df, signal_column):
    """
    Adds a 'pointpos' column to the DataFrame to 
    indicate the position of support and resistance points.
    
    Parameters:
    df (DataFrame): DataFrame containing the stock data 
                    with the specified SR column, 'Low', and 'High' columns.
    sr_column (str): The name of the column to consider 
                     for the SR (support/resistance) points.
    
    Returns:
    DataFrame: The original DataFrame with an additional 'pointpos' column.
    """
    def pointpos(row):
        if row[signal_column] == 2:
            return row['Low'] - row['Low']*0.06
        elif row[signal_column] == 1:
            return row['High'] + row['High']*0.06
        else:
            return np.nan

    df['pointpos'] = df.apply(lambda row: pointpos(row), axis=1)
    return df


In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

def plot_candlestick_with_signals(df, start_index, num_rows):
    """
    Plots a candlestick chart with signal points.
    
    Parameters:
    df (DataFrame): DataFrame containing the stock data with 
                    'Open', 'High', 'Low', 'Close', and 'pointpos' columns.
    start_index (int): The starting index for the subset of data to plot.
    num_rows (int): The number of rows of data to plot.
    
    Returns:
    None
    """
    df_subset = df[start_index:start_index + num_rows]
    
    fig = make_subplots(rows=1, cols=1)
    
    fig.add_trace(go.Candlestick(x=df_subset.index,
                                 open=df_subset['Open'],
                                 high=df_subset['High'],
                                 low=df_subset['Low'],
                                 close=df_subset['Close'],
                                 name='Candlesticks'),
                  row=1, col=1)
    
    # Flags to control legend display
    added_above_legend = False
    added_below_legend = False

    for i in range(len(df_subset)):
        if df_subset['pointpos'].iloc[i] > df_subset['High'].iloc[i]:
            # Plot downward triangle above the candle
            fig.add_trace(go.Scatter(x=[df_subset.index[i]], 
                                     y=[df_subset['pointpos'].iloc[i]], 
                                     mode="markers",
                                     marker=dict(size=12, 
                                                 color="MediumPurple", 
                                                 symbol='triangle-down'),
                                     name="Signal Above",
                                     showlegend=not added_above_legend),
                          row=1, col=1)
            # Disable further legend entries for above signals
            added_above_legend = True  

        elif df_subset['pointpos'].iloc[i] < df_subset['Low'].iloc[i]:
            # Plot upward triangle below the candle
            fig.add_trace(go.Scatter(x=[df_subset.index[i]], 
                                     y=[df_subset['pointpos'].iloc[i]], 
                                     mode="markers",
                                     marker=dict(size=12, 
                                                 color="MediumPurple", 
                                                 symbol='triangle-up'),
                                     name="Signal Below",
                                     showlegend=not added_below_legend),
                          row=1, col=1)
             # Disable further legend entries for below signals
            added_below_legend = True
    
    fig.update_layout(
        width=1200, 
        height=800, 
        plot_bgcolor='black',
        paper_bgcolor='black',
        font=dict(color='white'),
        xaxis=dict(showgrid=False, zeroline=False),
        yaxis=dict(showgrid=False, zeroline=False),
        showlegend=True,
        legend=dict(
            x=0.01,
            y=0.99,
            traceorder="normal",
            font=dict(
                family="sans-serif",
                size=12,
                color="white"
            ),
            bgcolor="black",
            bordercolor="gray",
            borderwidth=2
        )
    )
    
    fig.show()

In [ ]:
# Section 5.1
def collect_channel(candle, backcandles, window):
    """
    Collect pivot points data for the specified 
    candle index and calculate linear regressions.

    Parameters:
    candle (int): The current index of the candle.
    backcandles (int): Number of candles to look back.
    window (int): Window size to identify pivot points.

    Returns:
    tuple:  Slope and intercept of lows, 
            slope and intercept of highs, 
            and R-squared values for both.
    """
    local_df = df.iloc[candle - backcandles - window : candle - window].copy()
    local_df['isPivot'] = local_df.index.map(lambda idx: isPivot(idx, window))

    highs = local_df[local_df['isPivot'] == 1]['High']
    lows = local_df[local_df['isPivot'] == 2]['Low']

    if len(lows) >= 2 and len(highs) >= 2:
        sl_lows, interc_lows, r_value_l, _, _ = stats.linregress(lows.index, lows)
        sl_highs, interc_highs, r_value_h, _, _ = stats.linregress(highs.index, highs)

        return sl_lows, interc_lows, sl_highs, interc_highs, r_value_l**2, r_value_h**2

    return 0, 0, 0, 0, 0, 0


In [ ]:
def plot_candlestick(candle, backcandles, window):
    """
    Plots a candlestick chart with additional lines 
    based on the given candle, backcandles, and window parameters.

    Parameters:
    candle (int): The index of the candle to plot.
    backcandles (int): The number of candles to look back for calculations.
    window (int): The window size for rolling calculations.
    """

    fig = go.Figure(data=[go.Candlestick(x=dfpl.index,
                    open=dfpl['Open'],
                    high=dfpl['High'],
                    low=dfpl['Low'],
                    close=dfpl['Close'])])

    fig.add_scatter(x=dfpl.index, y=dfpl['pointpos'], mode="markers",
                    marker=dict(size=5, color="MediumPurple"),
                    name="pivot")

    sl_lows, interc_lows, sl_highs, interc_highs, r_sq_l, r_sq_h = collect_channel(candle, backcandles, window)
    print("R-squared values for low and high slopes are displayed above.", r_sq_l, r_sq_h)
    x = np.array(range(candle-backcandles-window, candle+1))
    fig.add_trace(go.Scatter(x=x, y=sl_lows*x + interc_lows, mode='lines', name='lower slope'))
    fig.add_trace(go.Scatter(x=x, y=sl_highs*x + interc_highs, mode='lines', name='max slope'))
    #fig.update_layout(xaxis_rangeslider_visible=False)
    fig.show()

plot_candlestick(76, 40, 3)


In [ ]:
# Section 5.2
    sl_lows_pct = (sl_lows / start_price_low) * 100
    sl_highs_pct = (sl_highs / start_price_high) * 100


In [ ]:
# Section 5.2
import numpy as np
from tqdm import tqdm

def calculate_slopes(df, candle, backcandles, window):
    """
    Calculate slopes and R-squared values for a given candle.
    
    Parameters:
    df (pd.DataFrame): The dataframe containing candle data.
    candle (int): The index of the candle to calculate for.
    backcandles (int): The number of candles to look back for calculations.
    window (int): The window size for rolling calculations.
    
    Returns:
    tuple: sl_lows_pct, sl_highs_pct, r_sq_l, r_sq_h
    """
    sl_lows, interc_lows, sl_highs, interc_highs, r_sq_l, r_sq_h = collect_channel(candle, backcandles, window)

    start_price_low = df['Low'].iloc[candle - backcandles - window]
    start_price_high = df['High'].iloc[candle - backcandles - window]

    sl_lows_pct = (sl_lows / start_price_low) * 100
    sl_highs_pct = (sl_highs / start_price_high) * 100

    return (sl_lows_pct, sl_highs_pct, 
            r_sq_l, r_sq_h, 
            sl_lows, interc_lows, 
            sl_highs, interc_highs)

def plot_triangle(df, candle, backcandles, 
                  window, slope_threshold, r2_threshold=0.9):
    """
    Plots a candlestick chart showing only the area where the triangle pattern is detected.

    Parameters:
    df (pd.DataFrame): The dataframe containing candle data.
    candle (int): The index of the candle where the triangle is detected.
    backcandles (int): The number of candles to look back for calculations.
    window (int): The window size for rolling calculations.
    slope_threshold (float): The threshold for detecting triangle patterns (in percentage).
    r2_threshold (float): The minimum R-squared value required for a valid fit (default is 0.9).
    """
    sl_lows_pct, sl_highs_pct, r_sq_l, r_sq_h, sl_lows, interc_lows, sl_highs, interc_highs = calculate_slopes(
        df, candle, backcandles, window
    )

    if (sl_highs_pct < -slope_threshold and 
        sl_lows_pct > slope_threshold and 
        r_sq_l >= r2_threshold and 
        r_sq_h >= r2_threshold):

        print(f"Triangle pattern detected at candle index {candle}.")

        # Find the first minima and maxima indices
        local_df = df.iloc[max(0, candle - backcandles - window):candle + 1]
        first_min_idx = local_df[local_df['Low'] == local_df['Low'].min()].index[0]
        first_max_idx = local_df[local_df['High'] == local_df['High'].max()].index[0]
        start_idx = min(first_min_idx, first_max_idx)

        # Define the range to plot
        plot_start_idx = max(0, start_idx - 50)
        plot_end_idx = min(len(df), candle + 1)

        # Slice the DataFrame to only include the relevant range
        df_slice = df.iloc[plot_start_idx:plot_end_idx]

        fig = go.Figure(data=[go.Candlestick(x=df_slice.index,
                                             open=df_slice['Open'],
                                             high=df_slice['High'],
                                             low=df_slice['Low'],
                                             close=df_slice['Close'])])

        fig.add_scatter(x=df_slice.index, y=df_slice['pointpos'], mode="markers",
                        marker=dict(size=5, color="MediumPurple"),
                        name="pivot")

        # Limit the x values for the lines to the detected triangle range
        x = np.array(range(start_idx, candle + 1))
        fig.add_trace(go.Scatter(x=x, y=sl_lows * x + interc_lows, 
                                 mode='lines', 
                                 name='lower slope (triangle)'))
        fig.add_trace(go.Scatter(x=x, y=sl_highs * x + interc_highs, 
                                 mode='lines', 
                                 name='upper slope (triangle)'))

        fig.show()
    else:
        print("No triangle pattern detected.")

def find_first_triangle(df, backcandles, window, slope_threshold=0.01, r2_threshold=0.9, start_index=None):
    """
    Loops through the DataFrame to find and plot the first triangle pattern.

    Parameters:
    df (pd.DataFrame): The DataFrame containing candlestick data.
    backcandles (int): The number of candles to look back for calculations.
    window (int): The window size for rolling calculations.
    slope_threshold (float): The threshold for detecting triangle patterns (in percentage).
    r2_threshold (float): The minimum R-squared value required for a valid fit (default is 0.9).
    start_index (int, optional): The index in the DataFrame where the search starts. 
                                 If None, it starts at `backcandles + window`.
    """
    if start_index is None:
        start_index = backcandles + window
    
    start_index = max(start_index, backcandles + window)
    
    for candle in tqdm(range(start_index, len(df))):
        sl_lows_pct, sl_highs_pct, r_sq_l, r_sq_h, *_ = calculate_slopes(df, candle, backcandles, window)
        
        if (sl_highs_pct < -slope_threshold and 
            sl_lows_pct > slope_threshold and 
            r_sq_l >= r2_threshold and 
            r_sq_h >= r2_threshold):
            plot_triangle(df, candle, backcandles, window, slope_threshold, r2_threshold)
            break
    else:
        print("No triangle pattern detected in the dataset.")


In [ ]:
# Section 5.2
def calculate_slopes(df, candle, backcandles, window):
    """
    Calculate slopes and R-squared values for a given candle, 
    ensuring that high and low pivots are not grouped
    together (e.g., avoiding sequences like low low high high).
    
    Parameters:
    df (pd.DataFrame): The dataframe containing candle data.
    candle (int): The index of the candle to calculate for.
    backcandles (int): The number of candles to look back for calculations.
    window (int): The window size for rolling calculations.
    
    Returns:
    tuple: sl_lows_pct, sl_highs_pct, 
    r_sq_l, r_sq_h, sl_lows, interc_lows, sl_highs, interc_highs
    """
    sl_lows, interc_lows, sl_highs, interc_highs, r_sq_l, r_sq_h = collect_channel(candle, backcandles, window)

    start_price_low = df['Low'].iloc[candle - backcandles - window]
    start_price_high = df['High'].iloc[candle - backcandles - window]

    sl_lows_pct = (sl_lows / start_price_low) * 100
    sl_highs_pct = (sl_highs / start_price_high) * 100

    # Access the isPivot column to check the sequence of pivots
    local_df = df.iloc[candle - backcandles - window:candle-window]
    pivots = [p for p in local_df['isPivot'].values if p not in [0, 3]]

    # Ensure no grouping of all highs or all lows
    if len(pivots) >= 3:
        if (all(p == 2 for p in pivots[:len(pivots)//2]) and 
            all(p == 1 for p in pivots[len(pivots)//2:])):
            return None, None, None, None, None, None, None, None
        if (all(p == 1 for p in pivots[:len(pivots)//2]) and 
            all(p == 2 for p in pivots[len(pivots)//2:])):
            return None, None, None, None, None, None, None, None

    return (sl_lows_pct, sl_highs_pct, 
            r_sq_l, r_sq_h, 
            sl_lows, interc_lows, 
            sl_highs, interc_highs)


In [ ]:
def find_first_triangle(df, backcandles, window, 
                        slope_threshold=0.01, r2_threshold=0.9, start_index=None):
    """
    Loops through the DataFrame to find and plot 
    the first triangle pattern, ensuring alternating high/low pivots.

    Parameters:
    df (pd.DataFrame): The DataFrame containing candlestick data.
    backcandles (int): The number of candles to look back for calculations.
    window (int): The window size for rolling calculations.
    slope_threshold (float): The threshold for detecting triangle patterns (in percentage).
    r2_threshold (float): The minimum R-squared value required for a valid fit (default is 0.9).
    start_index (int, optional): The index in the DataFrame where the search starts. 
                                 If None, it starts at `backcandles + window`.
    """
    if start_index is None:
        start_index = backcandles + window
    
    start_index = max(start_index, backcandles + window)
    
    for candle in tqdm(range(start_index, len(df))):
        results = calculate_slopes(df, candle, backcandles, window)
        
        if results[0] is None:  # if the calculation was skipped due to non-alternating pivots
            continue
        
        sl_lows_pct, sl_highs_pct, r_sq_l, r_sq_h, *_ = results
        
        if (sl_highs_pct < -slope_threshold and 
            sl_lows_pct > slope_threshold and 
            r_sq_l >= r2_threshold and 
            r_sq_h >= r2_threshold):
            plot_triangle(df, candle, backcandles, 
                          window, slope_threshold, r2_threshold)
            break
    else:
        print("No triangle pattern detected in the dataset.")


In [ ]:
# Section 6.1
def is_engulfing(df, candleidx, engulfdiff_pct=0.01):
    """
    Determine if a given candle is part of a bullish or bearish engulfing  pattern.

    Parameters:
    candleidx (int): The index of the current candle in the DataFrame.
    df (pandas.DataFrame): The DataFrame containing the OHLC (Open, High, Low, Close) data.
    engulfdiff_pct (float): The minimum percentage difference 
                            used to confirm an engulfing pattern.

    Returns:
    int: Returns 2 if a bullish engulfing pattern is detected, 
                 1 if a bearish engulfing pattern is detected, 
                 and 0 if neither pattern is found.
    """
    # Ensure we're not at the start of the DataFrame
    if candleidx < 1:
        return 0

    # Calculate the difference thresholds based on the 
    # percentage of the previous candle's closing price
    engulfdiff = df['Close'][candleidx - 1] * engulfdiff_pct/100

    # Check for a bullish engulfing pattern
    if (df['Close'][candleidx - 1] < df['Open'][candleidx - 1] and
        df['Close'][candleidx] > df['Open'][candleidx] and
        df['Close'][candleidx - 1] - df['Open'][candleidx] > engulfdiff and
        df['Open'][candleidx - 1] - df['Close'][candleidx] < -engulfdiff):
        return 2

    # Check for a bearish engulfing pattern
    if (df['Close'][candleidx - 1] > df['Open'][candleidx - 1] and
        df['Close'][candleidx] < df['Open'][candleidx] and
        df['Open'][candleidx] - df['Close'][candleidx - 1] > engulfdiff and
        df['Close'][candleidx] - df['Open'][candleidx - 1] < -engulfdiff):
        return 1

    # If no engulfing pattern is found
    return 0


In [ ]:
# Section 6.2
def is_engulfing(df, l, engulfdiff_pct=0.0001, engulfing_candles=1):
    """
    Determine if a given candle is an engulfing candle pattern.

    Parameters:
    df (DataFrame): DataFrame containing Open and Close prices.
    l (int): Current position in the DataFrame to evaluate.
    engulfdiff_pct (float, optional):   Percentage difference to consider 
                                        for engulfing. Default is 0.0001.
    engulfing_candles (int, optional): Number of candles to check for 
                                       the engulfing pattern. Default is 1.

    Returns:
    int: 0 if not engulfing, 1 if bearish engulfing, 2 if bullish engulfing.
    """
    if l < engulfing_candles:
        return 0

    # Check if all recent engulfing_candles are of same color 
    # and the candle just before is an opposite color
    if (not all((df.Open[l-i] > df.Close[l-i]) == (df.Open[l] > df.Close[l]) 
                for i in range(engulfing_candles)) or 
       not ( (df.Open[l-engulfing_candles] < df.Close[l-engulfing_candles]) 
            == (df.Open[l] > df.Close[l]) ) ):
        return 0

    engulfdiff = df.Close[l] * engulfdiff_pct

    # Check if last candle is green then check if its closing price is greater
    # than the open price of l-engulfing_candles by at least engulfdiff
    # and apply symmetrical configuration for bearish engulfing detection
    if (df.Close[l] > df.Open[l] and df.Close[l] >= 
        df.Open[l-engulfing_candles] + engulfdiff and
        df.Close[l-engulfing_candles] - df.Open[l-engulfing_candles+1] > 
        engulfdiff):
        return 2

    if (df.Close[l] < df.Open[l] and df.Close[l] <= 
        df.Open[l-engulfing_candles] - engulfdiff and
        df.Close[l-engulfing_candles] - df.Open[l-engulfing_candles+1] 
        < -engulfdiff):
        return 1

    return 0


In [ ]:
# Section 6.3
def is_rejection(df, current_candle, body_perc = 0.01, wick_ratio=3):
    """
    Determine if a given candle is a rejection candle based on body  percentages and wick ratios.

    Parameters:
    df (pandas.DataFrame): DataFrame containing the candlestick data.
    current_candle (str or int): Identifier for the current candle (index or label).
    body_perc (float, optional): Threshold for the minimum candle body percentage (default is 0.1).
    wick_ratio (float, optional): Arbitrary ratio value for wick comparison (default is 3).

    Returns:
    int: 2 if the candle is a bullish rejection, 1 if bearish rejection, 0 otherwise.
    """
    current_pos = df.index.get_loc(current_candle)
    current_price = df['Close'].iloc[current_pos]

    body_length_percentage = ( abs(df['Open'].iloc[current_pos] - df['Close'].iloc[current_pos]) / current_price * 100)
    lower_wick_percentage = ( (min(df['Open'].iloc[current_pos], df['Close'].iloc[current_pos])
                    -df['Low'].iloc[current_pos]) / current_price * 100 )
    upper_wick_percentage = ( (df['High'].iloc[current_pos] - 
                               max(df['Open'].iloc[current_pos], df['Close'].iloc[current_pos])) / current_price * 100 )

    # Buy conditions
    wick_to_body_ratio = lower_wick_percentage / body_length_percentage
    c1 = body_length_percentage > body_perc  # Minimum threshold for body  
                                               length in percentage
    c2 = lower_wick_percentage > upper_wick_percentage*wick_ratio
    c3 = wick_to_body_ratio > wick_ratio  # Arbitrary ratio value

    if c1 and c2 and c3:
        return 2

    # sell conditions
    wick_to_body_ratio = upper_wick_percentage / body_length_percentage
    c1 = body_length_percentage > body_perc # Minimum threshold for 
                                              body length in percentage
    c2 = upper_wick_percentage > lower_wick_percentage*wick_ratio
    c3 = wick_to_body_ratio > wick_ratio  # Arbitrary ratio value

    if c1 and c2 and c3:
        return 1

    return 0


# Chapter 6

# Section 2
def isPivot(candle, window):
    """
    function that detects if a candle is a pivot/fractal point
    args: candle index, window before and after candle to test if pivot
    returns: 1 if pivot high, 2 if pivot low, 3 if both and 0 default
    """
    if candle-window < 0 or candle+window >= len(df):
        return 0
    
    pivotHigh = 1
    pivotLow = 2
    for i in range(candle-window, candle+window+1):
        if df.iloc[candle].Low > df.iloc[i].Low:
            pivotLow=0
        if df.iloc[candle].High < df.iloc[i].High:
            pivotHigh=0
    if (pivotHigh and pivotLow):
        return 3
    elif pivotHigh:
        return pivotHigh
    elif pivotLow:
        return pivotLow
    else:
        return 0


In [ ]:
def get_support_resistance_levels(df, start_row=3, end_row=205):
    """
    Identify support and resistance levels based 
    on pivot candles within a specified range.

    Parameters:
    df (pandas.DataFrame):  The DataFrame containing the price 
                            data and the `isPivot` column.
    start_row (int): The starting row index for 
                     identifying pivot candles. Default is 3.
    end_row (int): The ending row index for 
                   identifying pivot candles. Default is 205.

    Returns:
    list: A list of tuples where each tuple contains:
        - The row index (int) where a support or 
          resistance level was identified.
        - The price level (float) at that row.
        - An indicator (int) where 2 represents 
          support and 1 represents resistance.
    """
    sr = []
    for row in range(start_row, end_row):  
        if df.isPivot[row] == 2:  # 2 indicates a support level
            sr.append((row, df.Low[row], 2))
        elif df.isPivot[row] == 1:  # 1 indicates a resistance level
            sr.append((row, df.High[row], 1))
    return sr


In [ ]:
def clean_support_resistance(sr, threshold_percent=0.5):
    """
    Clean the support and resistance levels by removing levels that are
    close to each other based on a specified threshold percentage.

    Parameters:
    sr (list of tuples): A list of tuples where each tuple contains:
        - The row index (int) where a support or resistance level was identified.
        - The price level (float) at that row.
        - An indicator (int) where 2 represents support and 1 represents resistance.
    threshold_percent (float): The threshold percentage to determine how close
                               the levels can be before being considered for removal.
                               Default is 0.5%.

    Returns:
    tuple: Two lists of tuples:
        - plotlist1: Cleaned resistance levels.
        - plotlist2: Cleaned support levels.
    """
    # Extract resistance and support levels with their original format
    plotlist1 = [(x[0], x[1], x[2]) for x in sr if x[2] == 1]  # Resistance levels
    plotlist2 = [(x[0], x[1], x[2]) for x in sr if x[2] == 2]  # Support levels

    # Sort the lists by the price level
    plotlist1.sort(key=lambda x: x[1])
    plotlist2.sort(key=lambda x: x[1])

    # Remove close resistance levels based on percentage threshold
    i = 1
    while i < len(plotlist1):
        current_level = plotlist1[i][1]
        previous_level = plotlist1[i - 1][1]
        threshold = threshold_percent / 100 * current_level

        if abs(current_level - previous_level) <= threshold:
            plotlist1.pop(i)
        else:
            i += 1

    # Remove close support levels based on percentage threshold
    i = 1
    while i < len(plotlist2):
        current_level = plotlist2[i][1]
        previous_level = plotlist2[i - 1][1]
        threshold = threshold_percent / 100 * current_level

        if abs(current_level - previous_level) <= threshold:
            plotlist2.pop(i)
        else:
            i += 1

    # Return the cleaned lists with the original format
    return plotlist1, plotlist2


In [ ]:
window=3
df['isPivot'] = df.apply(lambda x: isPivot(x.name,window), axis=1)

support_resistance_levels = get_support_resistance_levels(df, start_row=window, end_row=300)

cleaned_resistance, cleaned_support = clean_support_resistance(support_resistance_levels, threshold_percent=10)


In [ ]:
import plotly.graph_objects as go

def plot_candlestick_with_levels(df, start=0, end=200, support_levels=None, resistance_levels=None):
    """
    Plots a candlestick chart with support and resistance levels.

    Parameters:
    df (pandas.DataFrame): The DataFrame containing the OHLC data (Open, High, Low, Close).
    start (int): The starting index for the plot. Default is 0.
    end (int): The ending index for the plot. Default is 200.
    support_levels (list of tuples): A list of tuples representing support levels, 
                                     where each tuple contains the index, price level, and indicator.
    resistance_levels (list of tuples): A list of tuples representing resistance levels,
                                        where each tuple contains the index, price level, and indicator.

    Returns:
    None: Displays a candlestick chart with the specified support and resistance levels.
    """
    # Slice the DataFrame to the specified range
    dfpl = df[start:end]

    # Create the candlestick chart
    fig = go.Figure(data=[go.Candlestick(
        x=dfpl.index,
        open=dfpl['Open'],
        high=dfpl['High'],
        low=dfpl['Low'],
        close=dfpl['Close']
    )])

    # Add support levels as lines within the specified range
    if support_levels:
        for level in support_levels:
            if start <= level[0] < end:
                fig.add_shape(type='line',
                              x0=df.index[level[0]], y0=level[1],
                              x1=df.index[end-1], y1=level[1],
                              line=dict(color="MediumPurple", width=2)
                              )

    # Add resistance levels as lines within the specified range
    if resistance_levels:
        for level in resistance_levels:
            if start <= level[0] < end:
                fig.add_shape(type='line',
                              x0=df.index[level[0]], y0=level[1],
                              x1=df.index[end-1], y1=level[1],
                              line=dict(color="Green", width=2)
                              )

    # Display the chart
    fig.show()

In [ ]:
plot_candlestick_with_levels(df, start=0, end=100, support_levels=cleaned_support, resistance_levels=cleaned_resistance)

In [ ]:
# Section 3
def generate_signal(df, l, backcandles, gap, zone_threshold, price_diff_threshold):
    """
    Generates a trading signal based on EMA signals, 
    Fibonacci retracement levels, and price action.

    Parameters:
    df (pandas.DataFrame): The DataFrame containing the relevant 
                           market data (high, low, close prices, and EMASignal).
    l (int): The current index or position in the 
             DataFrame for which the signal is being generated.
    backcandles (int): The number of candles to look 
                       back to find the pivot points (high and low).
    gap (int): The number of candles to look forward 
               for confirming pivot points, avoiding lookahead bias.
    zone_threshold (float): The maximum allowed difference between 
                            the current close price and the calculated entry level (l1).
    price_diff_threshold (float): The minimum price difference required 
                            between the identified high and low for a valid signal.

    Returns:
    tuple: A tuple containing:
        - Signal type (2 for short, 1 for long, 0 for no signal)
        - Stop-loss level (float)
        - Take-profit level (float)
        - Index of the identified minimum price within the lookback window (int)
        - Index of the identified maximum price within the lookback window (int)
    """
    # Identify the maximum and minimum prices in the specified lookback window
    max_price = df.high[l-backcandles:l-gap].max()
    min_price = df.low[l-backcandles:l-gap].min()
    index_max = df.high[l-backcandles:l-gap].idxmax()
    index_min = df.low[l-backcandles:l-gap].idxmin()
    price_diff = max_price - min_price

    # Check for short signal
    if (df.EMASignal[l] == 2 and index_min < index_max and price_diff > price_diff_threshold):
        entry_level = max_price - 0.62 * price_diff  # Position entry at 0.62 Fibonacci level
        stop_loss = max_price - 0.78 * price_diff    # Stop-loss at 0.78 Fibonacci level
        take_profit = max_price                      # Take-profit at 0 Fibonacci level

        # Verify if the current close price is within the threshold zone and confirm entry conditions
        if abs(df.close[l] - entry_level) < zone_threshold and df.high[l-gap:l].min() > entry_level:
            return (2, stop_loss, take_profit, index_min, index_max)
        else:
            return (0, 0, 0, 0, 0)

    # Check for long signal
    elif (df.EMASignal[l] == 1 and index_min > index_max and price_diff > price_diff_threshold):
        entry_level = min_price + 0.62 * price_diff  # Position entry at 0.62 Fibonacci level
        stop_loss = min_price + 0.78 * price_diff    # Stop-loss at 0.78 Fibonacci level
        take_profit = min_price                      # Take-profit at 0 Fibonacci level

        # Verify if the current close price is within the threshold zone and confirm entry conditions
        if abs(df.close[l] - entry_level) < zone_threshold and df.low[l-gap:l].max() < entry_level:
            return (1, stop_loss, take_profit, index_min, index_max)
        else:
            return (0, 0, 0, 0, 0)

    # Return no signal if conditions are not met
    return (0, 0, 0, 0, 0)


In [ ]:
def apply_trading_signals(df, gap_candles=5, backcandles=40, 
                          zone_threshold=0.001, price_diff_threshold=0.01):
    """
    Applies trading signals to the DataFrame using the generate_signal function.

    Parameters:
    df (pandas.DataFrame): The DataFrame containing the market data.
    gap_candles (int):  The number of candles to use as a gap 
                        for pivot point detection. Default is 5.
    backcandles (int): The number of candles to look back 
                       for identifying pivot points. Default is 40.
    zone_threshold (float): The maximum allowed difference between the 
                            current closing price and the entry level. Default is 0.001.
    price_diff_threshold (float): The minimum price difference between identified 
                                  pivot points to validate a signal. Default is 0.01.

    Returns:
    pandas.DataFrame: The updated DataFrame with new columns for trading signals, 
                      stop-loss, take-profit, minimum swing, and maximum swing.
    """
    signal = [0] * len(df)
    TP = [0] * len(df)
    SL = [0] * len(df)
    MinSwing = [0] * len(df)
    MaxSwing = [0] * len(df)

    for row in range(backcandles, len(df)):
        gen_sig = generate_signal(df, row, backcandles=backcandles, gap=gap_candles, 
                                  zone_threshold=zone_threshold, 
                                  price_diff_threshold=price_diff_threshold)
        signal[row] = gen_sig[0]
        SL[row] = gen_sig[1]
        TP[row] = gen_sig[2]
        MinSwing[row] = gen_sig[3]
        MaxSwing[row] = gen_sig[4]
    
    df['signal'] = signal
    df['SL'] = SL
    df['TP'] = TP
    df['MinSwing'] = MinSwing
    df['MaxSwing'] = MaxSwing
    
    return df

df_with_signals = apply_trading_signals(df, gap_candles=10, 
                                        backcandles=50, 
                                        zone_threshold=0.002, 
                                        price_diff_threshold=0.02)

# Chapter 7

In [ ]:
# Section 1
import yfinance as yf
import pandas as pd
df = yf.download("BTC-USD", period='400d', interval='1d')
df=df[df["High"]!=df['Low']]
df.reset_index(inplace=True)

In [ ]:
# Section 1.1
def average_next_n_candles(df, i, N=10):
    """
    Calculate the average closing price of the next N candles.

    Parameters:
    df (DataFrame): The data frame containing the candle data.
    i (int): The index of the current candle.
    N (int): The number of candles to consider after the current one.

    Returns:
    float: The average closing price of the next N candles, or None if there are
           fewer than N candles remaining.
    """
    # Check if there are N candles after the current one
    if i + N >= len(df):
        return None

    # Compute the average closing price of the next N candles
    return df['Close'].iloc[i+1:i+N+1].mean()

In [ ]:
def average_next_n_mid_prices(df, i, N=10):
    """
    Calculate the average mid price (average of High and Low) of the next N candles.

    Parameters:
    df (DataFrame): The data frame containing the candle data.
    i (int): The index of the current candle.
    N (int): The number of candles to consider after the current one.

    Returns:
    float: The average mid price of the next N candles, or None if there are
           fewer than N candles remaining.
    """
    # Check if there are N candles after the current one
    if i + N >= len(df):
        return None

    # Compute the mid price for each of the next N candles
    mid_prices = (df['High'].iloc[i+1:i+N+1] + df['Low'].iloc[i+1:i+N+1]) / 2

    # Return the average of these mid prices
    return mid_prices.mean()

In [ ]:
df['future_average'] = df.apply(lambda row: average_next_n_candles(df, row.name, 10), axis=1)

In [ ]:
# Section 1.2
def category_next_n_candles(df, i, N=10, threshold_pct=1.0):
    """
    Calculate the average closing price of the next N candles and compare it 
    to the current candle's close price based on a percentage threshold.

    Parameters:
    df (DataFrame): The data frame containing the candle data.
    i (int): The index of the current candle.
    N (int): The number of candles to consider after the current one.
    threshold_pct (float): The percentage threshold to compare the future 
                           average price.

    Returns:
    int: 
        - 2 if the future average price is above the current close price by 
          more than the threshold.
        - 1 if the future average price is below the current close price by 
          more than the threshold.
        - 0 if the future average price is within the threshold range.
    """
    # Check if there are N candles after the current one
    if i + N >= len(df):
        return None

    # Compute the average closing price of the next N candles
    future_avg = df['Close'].iloc[i+1:i+N+1].mean()
    
    # Calculate the percentage difference between the current close and the future average price
    current_close = df['Close'].iloc[i]
    pct_diff = ((future_avg - current_close) / current_close) * 100

    # Return the appropriate value based on the percentage difference
    if pct_diff > threshold_pct:
        return 2
    elif pct_diff < -threshold_pct:
        return 1
    else:
        return 0

In [ ]:
df['future_cat'] = df.apply(lambda row: category_next_n_candles(df, row.name, 10, threshold_pct=2), axis=1)

In [ ]:
# Section 2
def test_signal_accuracy(df, signal_col, price_cat_col):
    """
    Calculate the accuracy of the signal column against 
    the price category column.

    Parameters:
    df (pd.DataFrame): The dataframe containing the data
    signal_col (str): The column name for signal
    price_cat_col (str): The column name for price category

    Returns:
    tuple: A tuple containing the percentage of 
           equal signals and different signals
    """
    equal_count = 0
    different_count = 0
    total_count = 0

    for i in range(len(df)):
        if df[signal_col].iloc[i] != 0:
            total_count += 1
            if df[signal_col].iloc[i] == df[price_cat_col].iloc[i]:
                equal_count += 1
            else:
                different_count += 1

    equal_percentage = (equal_count / total_count) * 100
    different_percentage = (different_count / total_count) * 100

    return equal_percentage, different_percentage

In [ ]:
# Section 4
df["future_diff"] = df["future_average"]-df["Close"]

In [ ]:
# Section 4.1
def calculate_descriptive_statistics(df, signal_col, num_col):
    """
    Calculate descriptive statistics for each category in the signal column.

    Parameters:
    df (DataFrame): The DataFrame containing the data.
    signal_col (str): The name of the categorical signal column.
    num_col (str): The name of the numerical column.

    Returns:
    DataFrame: Descriptive statistics for each category in the signal column.
    """
    return df.groupby(signal_col)[num_col].describe()

calculate_descriptive_statistics(df, "isRejection", "future_diff")

In [ ]:
# Section 4.2
from scipy.stats import f_oneway

def perform_anova_test(df, signal_col, num_col):
    """
    Perform a one-way ANOVA test to check if the means of the numerical column differ between signal categories.

    Parameters:
    df (DataFrame): The DataFrame containing the data.
    signal_col (str): The name of the categorical signal column.
    num_col (str): The name of the numerical column.

    Returns:
    dict: A dictionary with the F-statistic and p-value.
    """
    categories = df[signal_col].unique()
    category_data = [df[df[signal_col] == cat][num_col].dropna() for cat in categories]
    anova_result = f_oneway(*category_data)
    return {
        "F-statistic": anova_result.statistic,
        "p-value": anova_result.pvalue
    }
perform_anova_test(df, "isRejection", "future_diff")

In [ ]:
# Section 4.3
from scipy.stats import chi2_contingency

def perform_chi2_test(df, signal_col, num_col):
    """
    Perform a Chi-square test to examine the association between the categorical signal and numerical values.

    Parameters:
    df (DataFrame): The DataFrame containing the data.
    signal_col (str): The name of the categorical signal column.
    num_col (str): The name of the numerical column.

    Returns:
    dict: A dictionary with the Chi2-statistic, p-value, degrees of freedom, and expected frequencies.
    """
    df['num_category'] = pd.cut(df[num_col], bins=3, labels=[0, 1, 2])  # Discretize the numerical column into 3 bins
    contingency_table = pd.crosstab(df[signal_col], df['num_category'])
    chi2_result = chi2_contingency(contingency_table)
    return {
        "Chi2-statistic": chi2_result[0],
        "p-value": chi2_result[1],
        "degrees_of_freedom": chi2_result[2],
        "expected_frequencies": chi2_result[3]
    }
perform_chi2_test(df, "isRejection", "future_diff")

# Chapter 8

In [ ]:
# Section 5
def check_candles(df, backcandles, ma_column):
    """
    Categorize the trend based on the position of candle closes relative to a moving average.

    Parameters:
    df (DataFrame): The DataFrame containing the candle data.
    backcandles (int): The number of backcandles to consider for the trend calculation.
    ma_column (str): The name of the column representing the moving average.

    Returns:
    list: A list of trend categories where:
          - 2 indicates an uptrend (all close prices are above the moving average),
          - 1 indicates a downtrend (all close prices are below the moving average),
          - 0 indicates no clear trend.
    """
    categories = [0] * backcandles  # Initialize the first few elements as 0 (no trend)

    for i in range(backcandles, len(df)):
        close_prices = df['Close'][i-backcandles:i]
        ma_values = df[ma_column][i-backcandles:i]

        if all(close_prices > ma_values):
            categories.append(2)  # Uptrend
        elif all(close_prices < ma_values):
            categories.append(1)  # Downtrend
        else:
            categories.append(0)  # No trend

    return categories

# Apply the function to the DataFrame
df['Category'] = check_candles(df, 5, 'SMA_20')

In [ ]:
# Section 5.1
# Apply the function to the DataFrame
df['Trend_Category'] = check_candles(df, 5, 'SMA')


In [ ]:
# Section 5.1
def apply_entry_conditions(df, trend_col, upper_band_col, lower_band_col):
    """
    Apply entry conditions to the DataFrame and assign entry categories based on trend and Bollinger Bands.

    Parameters:
    df (DataFrame): The DataFrame containing the data.
    trend_col (str): The name of the column indicating the trend category.
    upper_band_col (str): The name of the column indicating the upper Bollinger Band.
    lower_band_col (str): The name of the column indicating the lower Bollinger Band.

    Returns:
    DataFrame: The DataFrame with an additional 'entry' column indicating the entry signals.
    """
    df['entry'] = 0

    # Buy entry condition
    buy_entry_condition = (df[trend_col] == 2) & (df['Open'] < df[lower_band_col]) & (df['Close'] > df[lower_band_col])
    df.loc[buy_entry_condition, 'entry'] = 2

    # Sell entry condition
    sell_entry_condition = (df[trend_col] == 1) & (df['Open'] > df[upper_band_col]) & (df['Close'] < df[upper_band_col])
    df.loc[sell_entry_condition, 'entry'] = 1

    return df

In [ ]:
df= apply_entry_conditions(df, "Trend_Category", "Upper Band", "Lower Band")

# Chapter 9

In [1]:
# Section 2
def check_candle_signal(df, candle_index, 
                        backcandles_window, pivot_window, 
                        distance_threshold_coef=0.003):
    """
    Determines if a candle is a valid rejection signal near support or resistance levels.

    Parameters:
    df (pd.DataFrame): The DataFrame containing the trading data.
    candle_index (int): The index of the current candle being analyzed.
    backcandles_window (int): The number of previous candles to 
                              consider for detecting support and resistance levels.
    pivot_window (int): The window size for detecting pivots, 
                        used to avoid lookahead bias.
    distance_threshold_coef (float): The coefficient to determine the proximity 
                                     threshold for a candle to be considered near a key level.

    Returns:
    int: 1 for a bearish rejection, 2 for a bullish rejection, 0 for no valid signal.
    """
    if candle_index < backcandles_window + pivot_window:
        return 0

    # Get and clean support/resistance levels from previous candles
    support_resistance_levels = get_support_resistance_levels(df, 
                                                              start_row=candle_index-backcandles_window, 
                                                              end_row=candle_index-pivot_window)
    
    resistance_levels, support_levels = clean_support_resistance(support_resistance_levels, 
                                                                 threshold_percent=1)
    
    # Flatten support and resistance levels
    key_levels = [x[1] for x in resistance_levels + support_levels]
    
    # Determine proximity threshold based on the current closing price
    close_distance_threshold = df.Close[candle_index] * distance_threshold_coef
    
    # Check if the current candle is near any resistance or support levels
    close_to_resistance = closeResistance(candle_index, 
                                          key_levels, 
                                          close_distance_threshold, 
                                          df)
    close_to_support = closeSupport(candle_index, 
                                    key_levels, 
                                    close_distance_threshold, 
                                    df)

    # Evaluate if the candle is a valid rejection signal
    if (df.rejection[candle_index] == 1 and 
        close_to_resistance and 
        is_below_resistance(candle_index, pivot_window, close_to_resistance, df) ):
        return 1  # Bearish rejection
    elif (df.rejection[candle_index] == 2 and 
          close_to_support and 
          is_above_support(candle_index, pivot_window, close_to_support, df) ):
        return 2  # Bullish rejection
    else:
        return 0  # No valid signal


In [ ]:
def closeResistance(l, levels, lim, df):
    if not levels:
        return 0
    nearest = min(levels, key=lambda x: abs(x - df['High'][l]))
    if (abs(df['High'][l] - nearest) <= lim or abs(max(df['Open'][l], df['Close'][l]) - nearest) <= lim) \
            and min(df['Open'][l], df['Close'][l]) < nearest \
            and df['Low'][l] < nearest:
        return nearest
    return 0

def closeSupport(l, levels, lim, df):
    if not levels:
        return 0
    nearest = min(levels, key=lambda x: abs(x - df['Low'][l]))
    if (abs(df['Low'][l] - nearest) <= lim or abs(min(df['Open'][l], df['Close'][l]) - nearest) <= lim) \
            and max(df['Open'][l], df['Close'][l]) > nearest \
            and df['High'][l] > nearest:
        return nearest
    return 0

In [ ]:
def is_below_resistance(l, backCandles, level, df):
    return df.loc[l-backCandles:l-1, 'High'].max() < level

def is_above_support(l, backCandles, level, df):
    return df.loc[l-backCandles:l-1, 'Low'].min() > level

In [ ]:
# Section 6
df.set_index(["Date"], inplace=True, drop=True)

from backtesting import Strategy, Backtest

def SIGNAL():
    return df.signal

class MyCandlesStrat(Strategy):  
    def init(self):
        super().init()
        self.signal1 = self.I(SIGNAL)
        self.ratio = 2
        self.risk_perc = 0.1

    def next(self):
        super().next() 
        if self.signal1==2:
            sl1 = self.data.Close[-1] - self.data.Close[-1]*self.risk_perc
            tp1 = self.data.Close[-1] + (self.data.Close[-1]*self.risk_perc)*self.ratio
            self.buy(sl=sl1, tp=tp1)
        elif self.signal1==1:
            sl1 = self.data.Close[-1] + self.data.Close[-1]*self.risk_perc
            tp1 = self.data.Close[-1] - (self.data.Close[-1]*self.risk_perc)*self.ratio
            self.sell(sl=sl1, tp=tp1)
bt = Backtest(df, MyCandlesStrat, cash=100000, margin=1/2, commission=0.02)
stat = bt.run()


In [ ]:
# Section 7

df['ATR'] = ta.atr(high=df.High, low=df.Low, close=df.Close, length=14)

In [ ]:
from backtesting import Strategy, Backtest
import pandas_ta as ta

df['ATR'] = ta.atr(high=df.High, low=df.Low, close=df.Close, length=14)

def SIGNAL():
    return df.signal

class MyCandlesStrat(Strategy): 
    atr_f = 2.5
    ratio_f = 1.5
    def init(self):
        super().init()
        self.signal1 = self.I(SIGNAL)

    def next(self):
        super().next() 
        if self.signal1==2:
            sl1 = self.data.Close[-1] - self.data.ATR[-1]*self.atr_f
            tp1 = self.data.Close[-1] + self.data.ATR[-1]*self.ratio_f*self.atr_f
            self.buy(sl=sl1, tp=tp1)
        elif self.signal1==1:
            sl1 = self.data.Close[-1] + self.data.ATR[-1]*self.atr_f
            tp1 = self.data.Close[-1] - self.data.ATR[-1]*self.ratio_f*self.atr_f
            self.sell(sl=sl1, tp=tp1)
bt = Backtest(df, MyCandlesStrat, cash=100000, margin=1/2, commission=0.02)
stat = bt.run()

In [ ]:
if self.signal1==2 and len(self.trades)==0:
            sl1 = self.data.Close[-1] - self.data.Close[-1]*self.risk_perc
            tp1 = self.data.Close[-1] + (self.data.Close[-1]*self.risk_perc)*self.ratio
            self.buy(sl=sl1, tp=tp1)
        elif self.signal1==1 and len(self.trades)==0:
            sl1 = self.data.Close[-1] + self.data.Close[-1]*self.risk_perc
            tp1 = self.data.Close[-1] - (self.data.Close[-1]*self.risk_perc)*self.ratio
            self.sell(sl=sl1, tp=tp1)

In [ ]:
# Section 8
class MyCandlesStrat(Strategy):
    def init(self):
        super().init()
        self.signal1 = self.I(SIGNAL)

    def next(self):
        super().next()
        sltr=self.data.Close[-1]*0.02

        for trade in self.trades: 
            if trade.is_long: 
                trade.sl = max(trade.sl or -np.inf, self.data.Close[-1] - sltr)
            else:
                trade.sl = min(trade.sl or np.inf, self.data.Close[-1] + sltr) 
        
        if self.signal1==2 and len(self.trades)==0: 
            sl1 = self.data.Close[-1] - sltr
            self.buy(sl=sl1)
        elif self.signal1==1 and len(self.trades)==0: 
            sl1 = self.data.Close[-1] + sltr
            self.sell(sl=sl1)

In [ ]:
sltr=self.data.Close[-1]*0.02

In [ ]:
for trade in self.trades: 
            if trade.is_long: 
                trade.sl = max(trade.sl or -np.inf, self.data.Close[-1] - sltr)
            else:
                trade.sl = min(trade.sl or np.inf, self.data.Close[-1] + sltr) 

In [ ]:
# Section 9
class MyStrat(Strategy):
    def init(self):
        super().init()
        self.signal = self.I(SIGNAL)
        self.mysize = 0.5

    def next(self):
        super().next()
        TPSLRatio = 2
        perc = 0.1

        if len(self.trades) == 1:
            self.trades[-1].sl = self.trades[-1].entry_price

        if self.signal == 2 and len(self.trades) == 0:
            sl1 = self.data.Close[-1] - self.data.Close[-1] * perc
            sldiff = abs(sl1 - self.data.Close[-1])
            tp1 = self.data.Close[-1] + sldiff * TPSLRatio
            tp2 = self.data.Close[-1] + sldiff
            self.buy(sl=sl1, tp=tp1, size=self.mysize)
            self.buy(sl=sl1, tp=tp2, size=self.mysize)

        elif self.signal == 1 and len(self.trades) == 0:
            sl1 = self.data.Close[-1] + self.data.Close[-1] * perc
            sldiff = abs(sl1 - self.data.Close[-1])
            tp1 = self.data.Close[-1] - sldiff * TPSLRatio
            tp2 = self.data.Close[-1] - sldiff
            self.sell(sl=sl1, tp=tp1, size=self.mysize)
            self.sell(sl=sl1, tp=tp1, size=self.mysize)

bt = Backtest(df, MyStrat, cash=100000, margin=0.5, commission=0.02)
stat = bt.run()

In [ ]:
if len(self.trades) == 1:
            self.trades[-1].sl = self.trades[-1].entry_price

In [ ]:
# Section 10
bt.plot()

# Chapter 10

In [ ]:
# Section 2
import numpy as np

def apply_blur(df, column_name, blur_percent):
    """
    Apply a normal distribution blur to a specified column in a DataFrame.

    Parameters:
    df (DataFrame): The DataFrame containing the data.
    column_name (str): The name of the column to apply the blur on.
    blur_percent (float): The percentage used to calculate the blur. This determines the range for generating new random values.

    Returns:
    DataFrame: A new DataFrame with the specified column blurred.
    """
    # Create a copy of the DataFrame to avoid modifying the original
    df_blurred = df.copy()

    for i in range(len(df_blurred)):
        original_value = df_blurred.at[i, column_name]
        std_dev = blur_percent * original_value / 3.0
        
        # Calculate the min and max bounds for the blur
        min_value = original_value - 3 * std_dev
        max_value = original_value + 3 * std_dev
        
        # Generate a new value using the normal distribution
        new_value = np.random.normal(loc=original_value, scale=std_dev)
              
        # Assign the new value to the DataFrame
        df_blurred.at[i, column_name] = new_value
    
    return df_blurred

In [ ]:
stats_list = []
for i in range(1000):
    window=3
    df_rand=df.copy()
    df_rand.reset_index(inplace=True)
    df_rand = apply_blur(df, "Close", 0.01)
    df_rand['rejection'] = df_rand.apply(lambda row: is_rejection(df_rand, row.name, body_perc = 0.1, wick_ratio=1.5), axis=1)
    df_rand['isPivot'] = df_rand.apply(lambda x: isPivot(x.name,window), axis=1)
    support_resistance_levels = get_support_resistance_levels(df_rand, start_row=window, end_row=300)
    cleaned_resistance, cleaned_support = clean_support_resistance(support_resistance_levels, threshold_percent=10)
    df_rand['signal'] = df_rand.apply(lambda x: check_candle_signal(df_rand, x.name, 60, 8), axis=1)
    df_rand.set_index(["Date"], inplace=True, drop=True)
    bt = Backtest(df_rand, MyCandlesStrat, cash=100000, margin=1/2, commission=0.02)
    stats_list.append(bt.run())

In [ ]:
# Section 3.1
import matplotlib.pyplot as plt

# Number of backtests
n_backtests = len(stats_list)

# Initialize a plot
plt.figure(figsize=(12, 6))

# Plot each equity curve
for i in range(n_backtests):
    equity_curve = stats_list[i]["_equity_curve"] # Get the equity curve DataFrame
    plt.plot(equity_curve.index, equity_curve['Equity'], label=f'Backtest {i+1}')

# Adding labels and title
plt.xlabel('Date or Time')
plt.ylabel('Equity')
plt.title('Equity Curves')

# Show the plot
plt.show()

In [ ]:
# Section 3.2
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import norm

def plot_distribution_with_normal_fit(stats_list, element_name):
    """
    Plots the distribution of a specified element from the backtest results
    and fits a normal distribution curve on top.

    Parameters:
    - stats_list: List of dictionaries containing backtest results.
    - element_name: The key of the element to plot (e.g., 'Return (Ann.) [%]').

    Returns:
    - A plot showing the distribution of the specified element with a normal distribution fit.
    """

    # Extract the specified element from each backtest
    element_values = [stats[element_name] for stats in stats_list]

    # Calculate mean and standard deviation for the normal distribution fit
    mean = np.mean(element_values)
    std_dev = np.std(element_values)

    # Plot the histogram of the element values
    plt.figure(figsize=(10, 6))
    sns.histplot(element_values, bins=10, kde=False, color='blue', stat='density', label=f'{element_name} Distribution')

    # Plot the normal distribution curve
    x = np.linspace(min(element_values), max(element_values), 100)
    p = norm.pdf(x, mean, std_dev)
    plt.plot(x, p, 'r--', label=f'Normal Fit (mean={mean:.2f}, std={std_dev:.2f})')

    # Adding labels and title
    plt.xlabel(element_name)
    plt.ylabel('Density')
    plt.title(f'Distribution of {element_name} with Normal Distribution Fit')
    plt.legend(loc='upper right')

    # Show the plot
    plt.show()

In [ ]:
plot_distribution_with_normal_fit(stats_list, 'Win Rate [%]')

In [ ]:
plot_distribution_with_normal_fit(stats_list, 'Return (Ann.) [%]')

In [ ]:
plot_distribution_with_normal_fit(stats_list, 'Return [%]')

In [ ]:
plot_distribution_with_normal_fit(stats_list, 'Sharpe Ratio')
plot_distribution_with_normal_fit(stats_list, 'Avg. Drawdown [%]')

# Chapter 11

# Section 2
import yfinance as yf

def get_data(symbol="AAPL"):
    # Define the ticker symbol
    ticker_symbol = symbol

    # Create a ticker object
    ticker = yf.Ticker(ticker_symbol)

    # Download historical data
    return ticker.history(period="2d", interval='1d')

historical_data = get_data()

In [ ]:
def test_engulfing(df):
    last_open = df.iloc[-1, :].Open
    last_close = df.iloc[-1, :].Close
    previous_open = df.iloc[-2, :].Open
    previous_close = df.iloc[-2, :].Close

    if (previous_open < previous_close 
        and last_open > previous_close 
        and last_close < previous_open):
        return 1  # Bearish Engulfing Pattern
    
    elif (previous_open > previous_close
          and last_open < previous_close 
          and last_close > previous_open):
        return 2  # Bullish Engulfing Pattern
    else:
        return 0  # No Engulfing Pattern

test_engulfing(historical_data)

In [ ]:
from apscheduler.schedulers.blocking import BlockingScheduler
from email.message import EmailMessage
import ssl
import smtplib
from creds import gmail_user, gmail_password

em = EmailMessage()

gmail_user = gmail_user
gmail_password = gmail_password
subject = 'info signal'

In [ ]:
def detection_job():
    # Initialize the message body with a generic introduction.
    msg = "Trading Signal Message \n"
    
    # Retrieve the most recent historical data (e.g., the last two candles).
    historical_data = get_data()
    
    # Check the historical data for an engulfing pattern.
    # If a bearish engulfing pattern is detected, update the message accordingly.
    if test_engulfing(historical_data) == 1:
        msg = str("The signal is 1: Bearish")
    
    # If a bullish engulfing pattern is detected, update the message accordingly.
    elif test_engulfing(historical_data) == 2:
        msg = str("The signal is 2: Bullish")

    # Set up the email details, specifying the sender, recipient, and subject.
    em['From'] = gmail_user
    em['To'] = gmail_user
    em['Subject'] = subject

    # Set the content of the email with the message that was created based on the signal.
    em.set_content(msg)

    # Create a secure SSL context to encrypt the email communication.
    context = ssl.create_default_context()

    # Connect to Gmail's SMTP server using SSL on port 465.
    server = smtplib.SMTP_SSL('smtp.gmail.com', 465, context=context)
    
    # Identify ourselves to the SMTP server.
    server.ehlo()

    # Log in to the Gmail account using the provided credentials.
    server.login(gmail_user, gmail_password)
    
    # Send the email to the same Gmail account (from `gmail_user` to `gmail_user`).
    server.sendmail(gmail_user, gmail_user, em.as_string())

    # Close the connection to the SMTP server.
    server.close()

In [ ]:
detection_job()

In [ ]:
scheduler = BlockingScheduler(job_defaults={'misfire_grace_time': 15*60})
scheduler.add_job(detection_job, 'cron', day_of_week='mon-fri', hour=0, minute=0, timezone=utc)
scheduler.start()

In [ ]:
# Section 3
symbols =  ['AAPL', 'NVDA', 'PYPL']

def some_job():
    msg="Trading Signal Message \n"
    for symb in symbols:
        historical_data = get_data(symb)
        if test_engulfing(historical_data)==1:
            msg = msg + str(symb+": the signal is 1 bearish") + "\n"

        elif test_engulfing(historical_data)==2:
            msg = msg + str(symb+": the signal is 2 bullish") + "\n"
    
    em['From'] = gmail_user
    em['To'] = gmail_user
    em['Subject'] = subject
    em.set_content(msg)

    context = ssl.create_default_context()

    server = smtplib.SMTP_SSL('smtp.gmail.com', 465, context=context)
    server.ehlo()
    server.login(gmail_user, gmail_password)
    server.sendmail(gmail_user, gmail_user, em.as_string())
    server.close()

In [ ]:
# Section 4
from apscheduler.schedulers.blocking import BlockingScheduler
from oandapyV20 import API
import oandapyV20.endpoints.orders as orders
from oandapyV20.contrib.requests import MarketOrderRequest
from oanda_candles import Pair, Gran, CandleClient
from oandapyV20.contrib.requests import TakeProfitDetails, StopLossDetails

In [ ]:
from config import access_token, accountID
access_token='Put Access Token Here'
accountID = 'Put Account ID Here'
def get_candles(n):
    client = CandleClient(access_token, real=False)
    collector = client.get_collector(Pair.BTC_USD, Gran.D1)
    candles = collector.grab(n)
    return candles

In [ ]:
def fetch_candle_data(num_candles=300):
    """
    Fetches the specified number of candles and returns a DataFrame.
    """
    candles = get_candles(num_candles)
    dfstream = pd.DataFrame(columns=['Open', 'Close', 'High', 'Low'])

    for i, candle in enumerate(candles):
        dfstream.loc[i, 'Open'] = float(candle.bid.o)
        dfstream.loc[i, 'Close'] = float(candle.bid.c)
        dfstream.loc[i, 'High'] = float(candle.bid.h)
        dfstream.loc[i, 'Low'] = float(candle.bid.l)

    return dfstream.astype(float)

In [ ]:
def execute_trade(client, accountID, signal, SLBuy, SLSell, TPBuy, TPSell):
    """
    Executes a trade based on the provided signal.
    """
    if signal == 1:  # Sell signal
        mo = MarketOrderRequest(
            instrument="BTC_USD",
            units=-1,
            takeProfitOnFill=TakeProfitDetails(price=TPSell).data,
            stopLossOnFill=StopLossDetails(price=SLSell).data
        )
    elif signal == 2:  # Buy signal
        mo = MarketOrderRequest(
            instrument="BTC_USD",
            units=1,
            takeProfitOnFill=TakeProfitDetails(price=TPBuy).data,
            stopLossOnFill=StopLossDetails(price=SLBuy).data
        )
    else:
        return

    r = orders.OrderCreate(accountID, data=mo.data)
    rv = client.request(r)
    print(rv)

In [ ]:
def trading_job():
    """
    Main trading function that fetches candle data, checks for a signal, and executes trades.
    """
    dfstream = fetch_candle_data()
    signal = check_candle_signal(
        l=len(dfstream) - 1,
        n1=6,
        n2=6,
        levelbackCandles=200,
        windowbackCandles=7,
        df=dfstream
    )

    SLBuy, SLSell, TPBuy, TPSell = calculate_sl_tp(dfstream)

    # Initialize API client
    client = API(access_token)

    # Execute trade based on the signal
    execute_trade(client, accountID, signal, SLBuy, SLSell, TPBuy, TPSell)

In [ ]:
scheduler = BlockingScheduler()
scheduler.add_job(trading_job, 'cron', hour='23', minute='55', start_date='2023-10-20 23:55:00', timezone='America/Chicago')
scheduler.start()